# Fooocus Easy Colab (run-all, DIVERSE faces preset)

1) Set Colab runtime to GPU.
2) Run all cells.
3) Open the printed link `✅ Open this URL`.

This notebook starts Fooocus in background and gives you a stable Colab proxy URL.
Default preset: `realistic_diverse` (reduced same-face bias, no auto-enhance/structure locks).
If needed, change `preset_name` in launch cell to `realistic_super` for stronger auto-enhancement.


In [ ]:
!pip install -q pygit2==1.15.1

%cd /content
!if [ ! -d "/content/Fooocus/.git" ]; then git clone -b cursor/git-eaf0 https://github.com/sunsideaspect/foocus_new.git Fooocus; fi
%cd /content/Fooocus
!git fetch origin cursor/git-eaf0
!git checkout cursor/git-eaf0
!git pull origin cursor/git-eaf0

%env TORCH_COMMAND=pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

import torch
print('Torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| cuda_available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime is required. Colab: Runtime -> Change runtime type -> GPU, then rerun.')


In [ ]:
import os
import subprocess
import sys
import time
import torch
from IPython.display import HTML, display
from google.colab import output

%cd /content/Fooocus

# Stop old instance if running
subprocess.run(["bash", "-lc", "pkill -f 'entry_with_update.py'"], check=False)

log_path = "/content/fooocus.log"
open(log_path, "w").close()

preset_name = "realistic_diverse"
total_vram_mb = 0
if torch.cuda.is_available():
    total_vram_mb = int(torch.cuda.get_device_properties(0).total_memory / (1024 * 1024))
vram_flag = "--always-high-vram" if total_vram_mb >= 20000 else "--always-normal-vram"

cmd = [
    sys.executable,
    "entry_with_update.py",
    "--preset", preset_name,
    "--listen", "0.0.0.0",
    "--port", "7865",
    vram_flag,
    "--disable-in-browser"
]

print(f"Preset: {preset_name}")
print(f"VRAM: {total_vram_mb} MB -> {vram_flag}")
print("Launching in background:", " ".join(cmd))
with open(log_path, "a") as f:
    proc = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT)

print(f"PID: {proc.pid}")


def try_get_proxy_url(port: int = 7865):
    try:
        return output.eval_js(f"google.colab.kernel.proxyPort({port})")
    except Exception:
        return None


proxy_url = None
print("Waiting for startup logs and proxy URL...")
started = False
for _ in range(60):
    txt = ""
    if os.path.exists(log_path):
        txt = open(log_path, "r", encoding="utf-8", errors="ignore").read()

    if "App started successful" in txt or "Running on local URL" in txt:
        started = True

    if proxy_url is None:
        proxy_url = try_get_proxy_url(7865)
        if proxy_url:
            print("\n✅ Open this URL:\n" + proxy_url + "\n")
            display(HTML(f'<a href="{proxy_url}" target="_blank">Open Fooocus (Colab proxy)</a>'))

    if proc.poll() is not None:
        break

    if started and proxy_url:
        break

    time.sleep(2)

if proc.poll() is not None:
    print(f"❌ Fooocus process exited with code {proc.returncode}. Check log below.")
elif started:
    print("✅ Server started.")
else:
    print("⏳ Still starting (first run downloads models, this can take several minutes).")

if proxy_url is None:
    print("\n⚠️ Could not get Colab proxy URL yet.")
    print("Reconnect runtime tab and rerun this launch cell once.")

print("\nLast log lines:")
subprocess.run(["bash", "-lc", f"tail -n 50 {log_path}"], check=False)


In [ ]:
# Optional: live logs
!tail -n 100 /content/fooocus.log
